In [10]:
import os
import statistics
from collections import Counter
from datetime import datetime
from json import dumps
import pandas as pd
import numpy as np
from pymongo import MongoClient
from dotenv import dotenv_values


In [11]:
# 1. Chargement ciblé des variables d'environnement
env_local_vars = dotenv_values(".env.local")
local_uri = env_local_vars.get("LOCAL_URI")

# 2. Test de la connexion locale
try:
    if not local_uri:
        raise ValueError("LOCAL_URI non trouvé dans .env.local")
    client = MongoClient(local_uri, serverSelectionTimeoutMS=5000)
    client.admin.command("ping")
    db = client["securite_routiere"]
    print("Connexion locale établie avec succès")
except Exception as e:
    print(f"Erreur de connexion locale : {e}")

Connexion locale établie avec succès


In [15]:

# 2. Chargement des fichiers CSV (ou récupération depuis les collections brutes)
df_caract = pd.read_csv("datasets/2024/caracteristiques-2024.csv", sep=";", low_memory=False)
df_lieux = pd.read_csv("datasets/2024/lieux-2024.csv", sep=";", low_memory=False)
df_vehicules = pd.read_csv("datasets/2024/vehicules-2024.csv", sep=";", low_memory=False)
df_usagers = pd.read_csv("datasets/2024/usagers-2024.csv", sep=";", low_memory=False)

# Remplacement des valeurs NaN par None (converti en null dans MongoDB)
for df in [df_caract, df_lieux, df_vehicules, df_usagers]:
    df.replace({np.nan: None}, inplace=True)

# 3. Nettoyage et préparation des données caractéristiques
def parse_date(row):
    try:
        hrmn = str(row["hrmn"]).zfill(4) if row["hrmn"] else "0000"
        return datetime(int(row["an"]), int(row["mois"]), int(row["jour"]), int(hrmn[:2]), int(hrmn[2:]))
    except Exception:
        return None

def parse_coord(val):
    if val is None:
        return None
    try:
        return float(str(val).replace(",", "."))
    except ValueError:
        return None

df_caract["date"] = df_caract.apply(parse_date, axis=1)
df_caract["lat_clean"] = df_caract["lat"].apply(parse_coord)
df_caract["long_clean"] = df_caract["long"].apply(parse_coord)

# 4. Structuration hiérarchique : Usagers -> Véhicules -> Accidents

# A. Dictionnaire des Usagers indexé par id_vehicule
usagers_by_vehicule = {}
cols_usager = [
    "id_usager", "place", "catu", "grav", "sexe", "An_nais",
    "trajet", "secu1", "secu2", "secu3", "locp", "actp", "etatp"
]
# Prise en compte du format secu1-3 si déjà agrégé
if "secu1-3" in df_usagers.columns:
    cols_usager = ["id_usager", "place", "catu", "grav", "sexe", "An_nais", "trajet", "secu1-3", "locp", "actp", "etatp"]

for row in df_usagers.to_dict(orient="records"):
    id_v = row.get("id_vehicule")
    doc_usager = {k: row[k] for k in cols_usager if k in row and row[k] is not None}
    usagers_by_vehicule.setdefault(id_v, []).append(doc_usager)

# B. Dictionnaire des Véhicules indexé par Num_Acc
vehicules_by_acc = {}
cols_vehicule = [
    "id_vehicule", "Num_Veh", "senc", "catv", "obs",
    "obsm", "choc", "manv", "motor", "occutc"
]

for row in df_vehicules.to_dict(orient="records"):
    num_acc = row.get("Num_Acc")
    id_v = row.get("id_vehicule")
    doc_vehicule = {k: row[k] for k in cols_vehicule if k in row and row[k] is not None}
    doc_vehicule["usagers"] = usagers_by_vehicule.get(id_v, [])
    vehicules_by_acc.setdefault(num_acc, []).append(doc_vehicule)

# C. Dictionnaire des Lieux indexé par Num_Acc
cols_lieu = [
    "catr", "voie", "V1", "V2", "circ", "nbv", "vosp", "prof",
    "pr", "pr1", "plan", "larrout", "surf", "infra", "situ", "vma"
]
lieux_by_acc = {}
for row in df_lieux.to_dict(orient="records"):
    num_acc = row.get("Num_Acc")
    doc_lieu = {k: row[k] for k in cols_lieu if k in row and row[k] is not None}
    lieux_by_acc[num_acc] = doc_lieu

# D. Construction finale des documents Accidents
cols_caract_env = ["adr", "dep", "com", "agg", "int", "atm", "lum", "col"]
accidents_docs = []

for row in df_caract.to_dict(orient="records"):
    num_acc = row.get("Num_Acc")
    doc_accident = {
        "Num_Acc": num_acc,
        "date": row.get("date"),
        "jour": row.get("jour"),
        "mois": row.get("mois"),
        "an": row.get("an"),
        "hrmn": str(row.get("hrmn")),
        "coordonnees": {
            "lat": row.get("lat_clean"),
            "long": row.get("long_clean")
        },
        "lieu": lieux_by_acc.get(num_acc, {}),
        "vehicules": vehicules_by_acc.get(num_acc, [])
    }
    
    for k in cols_caract_env:
        if k in row and row[k] is not None:
            doc_accident[k] = row[k]
            
    accidents_docs.append(doc_accident)

# 5. Écriture dans MongoDB
col_accidents = db["accidents"]
col_accidents.drop()  # Réinitialisation si la collection existe déjà

if accidents_docs:
    col_accidents.insert_many(accidents_docs)
    col_accidents.create_index("Num_Acc", unique=True)
    col_accidents.create_index("dep")
    print(f"{len(accidents_docs)} accidents insérés avec succès.")

54402 accidents insérés avec succès.


In [5]:
client.close()